# Sleep Learning Engine low-RAM cloud render

Run the full sleep_learning_engine pipeline on a free Google Colab T4 GPU. This is
the right path when your local machine runs out of memory during the
final encode: a 1080p H.264 video needs ~700 MB of free RAM for the
libx264 medium preset, and the per-pixel `geq` progress bar adds ~150
MB on top. Colab gives you 12.7 GB of system RAM plus a real NVIDIA
NVENC encoder, so the encode finishes in a couple of minutes.

**How to use:**
1. Click *Runtime -> Run all* (or press `Ctrl+F9`).
2. The upload cell will ask for your script text file and your
   background image - pick both files.
3. Wait for the render cell to finish (TTS, mix, encode).
4. The download cell will save the final MP4 to your machine.

**Cost:** free. Colab sessions cap at 12 hours and may disconnect if
idle, but a 6-minute video finishes in well under 10 minutes.

In [ ]:
# 1. Install sleep_learning_engine and its dependencies. Pulls from the public
# GitHub repo, so this only needs an internet connection.

# Purge pip's wheel cache first. Colab re-uses cached wheels across
# sessions, and a stale cache can serve the OLD package even after
# we push fixes to main. This is the fix for the 'I re-ran the
# notebook but the video did not change' report - the user was
# stuck on a cached wheel that did not include the legacy-toml
# fallback fix.
!pip cache purge
!pip install -q --force-reinstall https://github.com/fernandojjq/sleep-learning-engine/archive/refs/heads/main.tar.gz

# Print the installed version. If this still shows the old
# version, the cache purge + force-reinstall did not pick up the
# new wheel and the rest of the notebook will use stale code.
!pip show sleep_learning_engine | grep -E "^(Version|Location):"

# Verify the runtime actually has a GPU. The previous version only
# checked the ffmpeg encoder list, which lies: h264_nvenc can be in
# the list even when no GPU is bound to the container, and the user
# gets a misleading 'NVENC available:' with no follow-up text.
# The real test is `nvidia-smi` (raises if no GPU is present).
import subprocess
smi = subprocess.run(
    ["nvidia-smi", "--query-gpu=name,memory.total,driver_version",
     "--format=csv,noheader"], capture_output=True, text=True, timeout=15
)
if smi.returncode == 0 and smi.stdout.strip():
    print("GPU detected:")
    print(smi.stdout)
    print("Note: NVENC does NOT use VRAM. 0.0/15.0 GB USED is normal -",
          "NVENC runs on dedicated T4 silicon, not on the GPU cores.",
          "If VRAM shows >0 GB, the encode is going through the GPU",
          "and is therefore hardware-accelerated.")
else:
    print("=" * 60)
    print("NO GPU DETECTED in this Colab runtime.")
    print("Encode will fall back to libx264 (CPU), 5-10x slower.")
    print("")
    print("Fix: Runtime -> Change runtime type -> T4 GPU -> Save.")
    print("Reconnect when prompted. Re-run this cell.")
    print("=" * 60)

nvenc = subprocess.run(
    ["ffmpeg", "-hide_banner", "-encoders"], capture_output=True, text=True
).stdout
if "h264_nvenc" in nvenc:
    print("NVENC encoder: AVAILABLE")
else:
    print("NVENC encoder: NOT in ffmpeg build (libx264 fallback)")

# Replace Colab's system ffmpeg 4.4.2 with a modern static build.
# The system ffmpeg is from Ubuntu 22.04 (released 2021) and has
# known issues with the audio filter graph the studio uses
# (sidechaincompress + alimiter + concat demuxer) - the filter
# graph parser bails out with 'Stream specifier matches no
# streams' on that version. The static build is NVENC-capable,
# has every audio filter we need, and is what the rest of the
# notebook will find on PATH after this cell runs. ~80 MB
# download, ~30 s on a typical Colab connection.
import subprocess
ffmpeg_version = subprocess.run(
    ["ffmpeg", "-version"], capture_output=True, text=True
).stdout
if "ffmpeg version 4." in ffmpeg_version:
    print("Upgrading ffmpeg (Colab system is 4.4.2, too old for the studio filter graph)...")
    ffmpeg_install_cmd = (
        "set -e; "
        "curl -sL https://github.com/BtbN/FFmpeg-Builds/releases/download/latest/ffmpeg-master-latest-linux64-gpl.tar.xz "
        "| tar xJ; "
        "cp ffmpeg-master-latest-linux64-gpl/bin/ffmpeg /usr/local/bin/ffmpeg; "
        "cp ffmpeg-master-latest-linux64-gpl/bin/ffprobe /usr/local/bin/ffprobe; "
        "chmod +x /usr/local/bin/ffmpeg /usr/local/bin/ffprobe"
    )
    subprocess.check_call(['bash', '-c', ffmpeg_install_cmd])
    print("ffmpeg upgraded.")
else:
    print("ffmpeg is already modern enough.")
print(subprocess.run(
    ["ffmpeg", "-version"], capture_output=True, text=True
).stdout.split('\n')[0])

# === CONFIG: edit this dict and re-run to change the render. ===
# The cell writes the values to /content/.sleeplens.toml and
# exports SLEEP_LEARNING_ENGINE_HOME so the CLI finds it.
# Default values are the user's tuned Colab baseline (Brian voice,
# -20% rate, ambient at 0.54). Edit the dict and re-run to override.
# Full list of Edge TTS voice ids: https://learn.microsoft.com/
CONFIG = {
    "tts_voice": "en-US-BrianNeural",  # Edge TTS voice id
    "tts_rate": "-20%",                  # Edge TTS rate (e.g. -10% slower)
    "tts_pitch": "-2Hz",                 # Edge TTS pitch (e.g. -4Hz lower)
    "ambient_volume": 0.99,             # 0.0 (silent) to 1.0 (max)
    "ambient_duck_db": 8.0,            # dB cut on the bed while voice speaks
    "pause_between_paragraphs": 0.0,     # 0 = none; 1.8 was the old buggy default
    "language": "en",                    # script + voice language
    "output_preset": "sleep_1080p",      # sleep_720p or sleep_1080p or audio_only
    "hardware_accel": "auto",           # auto | nvenc | qsv | amf | libx264
    "render_threads": 1,                # 0 = auto, 1 = safest on shared Colab
}

import os
WORK = "/content"
os.makedirs(WORK, exist_ok=True)
config_path = f"{WORK}/.sleeplens.toml"
toml_lines = [
    f"# Generated by the notebook - edit the CONFIG dict above and re-run this cell.",
]
for k, v in CONFIG.items():
    if isinstance(v, str):
        toml_lines.append(f'{k} = "{v}"')
    elif isinstance(v, bool):
        toml_lines.append(f"{k} = {str(v).lower()}")
    else:
        toml_lines.append(f"{k} = {v}")
with open(config_path, "w", encoding="utf-8") as fh:
    fh.write("\n".join(toml_lines) + "\n")

print(f"Config written to {config_path}")
for k, v in CONFIG.items():
    print(f"  {k:<24}: {v}")

# Pin the project root to /content so load_settings finds the toml.
os.environ["SLEEP_LEARNING_ENGINE_HOME"] = WORK

# VOICE / RATE etc. are also exposed as plain variables for the
# render cell to use in shell commands without re-reading the toml.
VOICE = CONFIG["tts_voice"]

In [ ]:
# 2a. (Optional) Generate the script in Colab from a topic.
# Skip this cell entirely if you already have a script file - the
# next cell will pick it up from the upload widget.
#
# The script uses NVIDIA NIM with DeepSeek V4 Flash by default
# (free tier, no credit card). Swap base_url + model for OpenAI,
# Anthropic-via-proxy, or any other OpenAI-compatible endpoint.
import os

GENERATE_SCRIPT = False  # flip to True to enable in-Colab script generation
TOPIC = ""  # e.g. "how transformers attend to context"
TARGET_WORDS = 4500
API_KEY = ""  # your NVIDIA NIM key (or OpenAI key, or ...)
BASE_URL = "https://integrate.api.nvidia.com/v1"
MODEL = "deepseek-ai/deepseek-v4-flash"

if GENERATE_SCRIPT:
    if not TOPIC or not API_KEY:
        raise SystemExit("Set TOPIC and API_KEY before flipping GENERATE_SCRIPT to True.")
    from openai import OpenAI
    client = OpenAI(base_url=BASE_URL, api_key=API_KEY, timeout=180.0)
    from sleep_learning_engine.ai.script_writer import SYSTEM_PROMPT
    user_prompt = f"Topic: {TOPIC}\nTarget word count: {TARGET_WORDS}."
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_prompt},
        ],
        temperature=0.7, max_tokens=8192,
    )
    generated = response.choices[0].message.content
    os.makedirs("/content/assets", exist_ok=True)
    safe_topic = TOPIC.replace(" ", "_")[:40]
    SCRIPT = f"/content/assets/{safe_topic}.txt"
    with open(SCRIPT, "w", encoding="utf-8") as fh:
        fh.write(generated)
    print(f"Generated {len(generated.split())} words -> {SCRIPT}")
else:
    print("Skipped. Upload a .txt file in the next cell.")

In [ ]:
# 2b. Upload your script, background visual, AND ambient music.
# All three are sent through the browser, stay on Colab's VM,
# and are deleted when the session ends.
#
# Required uploads:
#   - 1 script text file (.txt) - the narration text
#   - 1 background image (.jpg / .png) or short video (.mp4)
#   - 3+ ambient .mp3 files - the music bed
#
# The package does NOT bundle any ambient music. You must
# supply your own (Minimax Music 2.6, Udio, Suno, Stable Audio,
# or any royalty-free source you have a license for).
from google.colab import files
import os, shutil

os.makedirs("/content/assets", exist_ok=True)
os.makedirs("/content/output", exist_ok=True)
os.makedirs("/content/assets/ambient", exist_ok=True)

print("Upload: 1 .txt script, 1 background image/video, 3+ ambient .mp3 files...")
uploaded = files.upload()

# Route each upload to the right folder.
script_uploaded = None
bg_uploaded = None
ambient_uploaded = 0
for name, data in uploaded.items():
    ext = os.path.splitext(name)[1].lower()
    if ext == ".txt":
        with open(f"/content/assets/{name}", "wb") as fh:
            fh.write(data)
        script_uploaded = f"/content/assets/{name}"
    elif ext in (".mp3", ".ogg", ".wav", ".flac", ".m4a", ".aac"):
        with open(f"/content/assets/ambient/{name}", "wb") as fh:
            fh.write(data)
        ambient_uploaded += 1
    elif ext in (".mp4", ".mov", ".webm", ".jpg", ".jpeg", ".png"):
        with open(f"/content/assets/{name}", "wb") as fh:
            fh.write(data)
        bg_uploaded = f"/content/assets/{name}"

# Validate: script + background + at least 3 ambient tracks.
missing = []
if not script_uploaded:
    missing.append("script .txt")
if not bg_uploaded:
    missing.append("background image or video")
if ambient_uploaded < 3:
    missing.append(f"at least 3 ambient .mp3 files (you uploaded {ambient_uploaded})")
if missing:
    print("=" * 60)
    print("MISSING REQUIRED UPLOADS:")
    for m in missing:
        print(f"  - {m}")
    print("")
    print("Re-run this cell and upload all three kinds of files at once.")
    print("=" * 60)
    raise FileNotFoundError("Missing required uploads - see messages above.")

# Heuristics for which file is which (re-using the names that came in).
import glob
SCRIPT = script_uploaded
BG_IMAGE = ""
BG_VIDEO = ""
if bg_uploaded.lower().endswith((".mp4", ".mov", ".webm")):
    BG_VIDEO = bg_uploaded
elif bg_uploaded.lower().endswith((".jpg", ".jpeg", ".png")):
    BG_IMAGE = bg_uploaded

print(f"Script:      {SCRIPT or "<missing>"}")
print(f"Background:  {BG_IMAGE or BG_VIDEO or "<missing>"}")
print(f"  is video:  {bool(BG_VIDEO)}")
print(f"Ambient:     {ambient_uploaded} tracks in /content/assets/ambient/")

if not SCRIPT:
    raise SystemExit("Need a .txt script. Either run the cell above to generate one, or upload a .txt here.")
if not (BG_IMAGE or BG_VIDEO):
    raise SystemExit("Need a background image (.jpg/.png) or a video (.mp4/.mov).")

In [ ]:
# 3. Render. The ffmpeg encode uses real NVENC on the T4, so a
# 6-minute 1080p video finishes in about 1-2 minutes.
import os, subprocess, sys

# On Colab, ffmpeg spawned as a subprocess from Python does NOT
# inherit the LD_LIBRARY_PATH that points at the NVIDIA driver
# libraries. Without them, the NVENC encoder fails silently or
# falls back to a software emulation that is 5-10x slower. Patch
# the env before the subprocess.Popen call.
NVIDIA_LIB_PATHS = [
    "/usr/lib64-nvidia",          # Colab default
    "/usr/lib/x86_64-linux-gnu",  # Debian / Ubuntu
    "/usr/local/cuda/lib64",      # Manual CUDA install
]
nvidia_path = next((p for p in NVIDIA_LIB_PATHS if os.path.exists(p)), None)
if nvidia_path:
    print(f"NVIDIA libs: patching LD_LIBRARY_PATH to include {nvidia_path}")
    os.environ["LD_LIBRARY_PATH"] = nvidia_path + ":" + os.environ.get("LD_LIBRARY_PATH", "")

OUT_STEM = "sleep_learning_engine-colab"
cmd = [
    "python", "-m", "sleep_learning_engine", "render",
    "--script", SCRIPT,
    "--output-stem", OUT_STEM,
]
if BG_IMAGE:
    cmd += ["--background-image", BG_IMAGE]
if BG_VIDEO:
    cmd += ["--background-video", BG_VIDEO]

# Popen (not run) so each ffmpeg / sleep_learning_engine progress
# line is printed to the cell output as it arrives. The previous
# version used subprocess.run(..., capture_output=True) which
# buffered the entire 10-minute render and looked frozen.
#
# SLEEP_LEARNING_ENGINE_HOME points the pipeline at /content so
# the resolved ambient_dir and output_dir align with the paths
# this notebook writes to (assets/ and output/). Without it, the
# auto-detect walks up from the pip-installed package and lands in
# /usr/local/lib/python3.12/dist-packages/.../assets/ambient (which
# is empty in the pip package) and the same for output (which is
# also not writable by the user).
#
# SLEEP_LEARNING_ENGINE_TTS_VOICE lets the user swap voice by
# editing the single VOICE line at the top of cell 1.
env = {
    **os.environ,
    "SLEEP_LEARNING_ENGINE_HOME": "/content",
    "SLEEP_LEARNING_ENGINE_TTS_VOICE": VOICE,
}
if nvidia_path:
    env["LD_LIBRARY_PATH"] = os.environ["LD_LIBRARY_PATH"]

# Defensive: re-write the toml from the CONFIG dict right before
# the render runs. Idempotent - protects against 'I edited CONFIG
# in cell 1 but the toml never refreshed' and 'I only re-ran this
# cell and CONFIG was stale'.
# If cell 1 was skipped entirely, CONFIG is undefined and we fail
# with a clear error pointing back at cell 1 instead of silently
# using package defaults (which is what produced 'the video did
# not change' on the last debug round).
if "CONFIG" not in globals():
    raise SystemExit(
        "CONFIG dict is not defined. Re-run cell 1 to write the "
        "toml and export SLEEP_LEARNING_ENGINE_HOME, then re-run "
        "this cell."
    )
toml_lines = [
    "# Re-emitted by cell 4 right before the render runs.",
]
for k, v in CONFIG.items():
    if isinstance(v, str):
        toml_lines.append(f'{k} = "{v}"')
    elif isinstance(v, bool):
        toml_lines.append(f"{k} = {str(v).lower()}")
    else:
        toml_lines.append(f"{k} = {v}")
config_path = f"{WORK}/.sleeplens.toml"
with open(config_path, "w", encoding="utf-8") as fh:
    fh.write("\n".join(toml_lines) + "\n")

# Cat the toml so the user can see exactly what the render uses.
print(f"Project root : {WORK}")
print(f"Config toml  : {config_path}")
print("Active settings that the CLI will read:")
with open(config_path, "r", encoding="utf-8") as fh:
    print(fh.read(), end="")
print(f"Voice        : {VOICE}")
print(f'Hardware     : {"NVENC" if nvidia_path else "libx264"}')
print("Running      :", " ".join(cmd))
print("-" * 60)
print("STREAMING OUTPUT (will return when the render finishes):")
print("-" * 60)

# subprocess.run with capture_output=True. ffmpeg's stdout and
# stderr are buffered until ffmpeg exits, then we print them.
# No Popen, no for-line loop, no threads, no deadlock possible.
result = subprocess.run(
    cmd,
    capture_output=True,
    text=True,
    env=env,
    timeout=4 * 60 * 60,
)
if result.stdout:
    print(result.stdout)
if result.stderr:
    print(result.stderr[-4000:])
print("-" * 60)
if result.returncode != 0:
    raise SystemExit(f"Render failed with exit code {result.returncode}. See stderr above.")
print("Render finished.")

In [ ]:
# 4. Download the final MP4. Colab saves it to /content/output/ and
# hands it to the browser, which triggers a normal download.
from google.colab import files
import glob, os, time

# IMPORTANT: the CLI calls paths.unique_output() which NEVER
# overwrites - the second render goes to OUT_STEM-1.mp4, the
# third to OUT_STEM-2.mp4, etc. The previous version of this
# cell used glob('OUT_STEM.mp4') which only matched the FIRST
# render's file. On every re-run, the user downloaded the OLD
# MP4 and assumed the new render was identical to the old one
# ('el video no cambio, mismo peso').
# Fix: match OUT_STEM*.mp4 and pick the newest by mtime.
candidates = sorted(glob.glob(f"/content/output/{OUT_STEM}*.mp4"), key=os.path.getmtime, reverse=True)
if not candidates:
    raise SystemExit("No MP4 was produced. Check the render cell for errors.")
output = candidates[0]
print("MP4s in output dir (newest first):")
for c in candidates:
    mtime = time.strftime("%H:%M:%S", time.localtime(os.path.getmtime(c)))
    size_mb = os.path.getsize(c) / 1e6
    marker = " <- downloading this one" if c == output else ""
    print(f"  {mtime}  {size_mb:6.1f} MB  {os.path.basename(c)}{marker}")
print(f"Final video: {output} ({os.path.getsize(output)/1e6:.1f} MB)")
files.download(output)

## What just happened

- **Cell 1** installed sleep_learning_engine from the public repo and confirmed
  Colab's ffmpeg build exposes NVENC (it always does on T4 instances).
- **Cell 2a** optionally generated the script in Colab from a topic
  using whatever API key you pasted in. Skip it if you uploaded a
  pre-written script.
- **Cell 2b** took the files you uploaded and resolved the script path,
  the background image, and the background video (whichever you gave).
  Either an image or a short loopable video works.
- **Cell 3** ran the full sleep_learning_engine CLI: TTS via Edge, mix with the
  procedural ambient bed, encode with `h264_nvenc` on the T4.
- **Cell 4** downloaded the MP4 to your machine.

## Why is the render taking longer than 1-2 minutes?

A 6-minute 1080p video should finish in **under 5 minutes** with the
T4 GPU. If it takes 10-15 minutes, one of these is happening:

- **GPU not enabled.** Open *Runtime -> Change runtime type* and pick
  *T4 GPU* under Hardware accelerator. This is the most common cause.
  Without GPU, ffmpeg falls back to `libx264` CPU encoding, which is
  5-10x slower.
- **Edge TTS is rate-limiting you.** Microsoft throttles to about 1
  request per second on the free service. A 35-segment script then
  takes 35+ seconds for the TTS phase alone. There is no workaround
  in the free tier; switch to a paid TTS backend if you hit this often.
- **Colab assigned a small VM.** Free tier GPUs are best-effort; if
  the assigned instance has less RAM, the encode may spill to swap.
  Rerun the cell and Colab will usually pick a better VM.

## Limitations of the free tier

- Colab free sessions cap at 12 hours and may disconnect when idle.
- GPU availability is not guaranteed; if no T4 is free, the notebook
  falls back to `libx264` software encoding (still 4-6x faster than a
  low-RAM Windows box because Colab has 12.7 GB free).
- The Edge TTS service is rate-limited; very long scripts may need
  to be split into chunks.
- Background videos up to 4K are supported but consume proportionally
  more memory during the loop step; 1080p is the sweet spot.

## Troubleshooting

If the render cell says `Render failed with exit code -12`, Colab ran
out of memory. Switch the runtime to a 25 GB RAM instance (Colab Pro)
or rerun the notebook - the failure is usually transient.